# Comparing classifiers, multi-class and imbalanced data

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## A guide table
| Model | Scaling | Boundary | Good probabilities | Prediction speed | Big data |
|---|---|---|---|---|---|
| Logistic | **Yes** | Linear | Excellent | Very fast | Excellent |
| KNN | **Yes** | Irregular | Medium | Slow | Poor |
| Naive Bayes | No | Simple | Poor | Very fast | Excellent |
| Tree | No | Steps | Poor | Very fast | Good |
| Forest | No | Soft steps | Good | Medium | Good |
| Boosting | No | Complex | Good | Fast | Excellent |
| Kernel SVM | **Yes** | Smooth curve | With tuning | Medium | Poor |

[[lab:cls_all]]

## Multi-class problems
Binary models (such as SVM) become multi-class with two strategies:

| Strategy | Models for K classes | Idea |
|---|---|---|
| One-vs-rest (OvR) | K | Each class against all others |
| One-vs-one (OvO) | K(K−1)/2 | Each pair of classes; a vote |
| Softmax | 1 | Multinomial logistic, directly |



In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import cross_val_score
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)   # 10 classes
for name, m in [("OvR  (10 models)", OneVsRestClassifier(LinearSVC(C=0.05))),
                ("OvO  (45 models)", OneVsOneClassifier(LinearSVC(C=0.05))),
                ("Softmax (1 model)", LogisticRegression(max_iter=2000))]:
    s = cross_val_score(make_pipeline(StandardScaler(), m), X, y, cv=5).mean()
    print(f"{name}: accuracy = {s:.3f}")



## Imbalanced data
When one class is rare (1% fraud):

| Remedy | How |
|---|---|
| A suitable metric | F1, recall, precision-recall AUC instead of accuracy |
| Class weights | `class_weight="balanced"` |
| Move the threshold | A threshold below 0.5 for more recall |
| Resampling | Shrink the big class or grow the small one (SMOTE in imbalanced-learn) |
| Stratified splits | `stratify=y` and `StratifiedKFold` |



In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X, y = make_classification(5000, n_features=10, weights=[0.97], flip_y=0.01, random_state=0)
print("Share of class 1:", y.mean().round(3))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw, max_iter=1000).fit(X_tr, y_tr)
    print("class_weight =", cw)
    print(classification_report(y_te, m.predict(X_te), digits=3))



> Never resample before splitting the data; only on the training data, inside cross-validation.